# FAseg — Batch Inference on Ex-Vivo Data (1 run)

Runs inference for the **Full pretraining × best_s3 fine-tuned model** on
ex-vivo beef leg data (`manual_seg_beef`).  Click "Run All".

Each run saves to `outputs/inference_results/exvivo/<exp>_<strategy>/`:
- `pred_mask_3d.npy` / `.bin` — 3-D binary mask
- `boundary_matrix.npy` — TOF boundary per row/slice
- `inference_time.txt`


In [1]:
import os, sys, json, time
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

_NB_DIR = os.path.abspath('')
if os.path.basename(_NB_DIR) == 'scripts':
    _project_root = os.path.dirname(_NB_DIR)
else:
    _project_root = _NB_DIR
_src_root = os.path.join(_project_root, 'src')
if _src_root not in sys.path:
    sys.path.insert(0, _src_root)

from faseg.models import UNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


In [2]:
# =========================================================================
# Config — fine-tuning checkpoint to run inference on (ex-vivo)
# =========================================================================
DATA_DIR = os.path.join(_project_root, 'manual segmentation', 'manual_seg_beef')

# (pretrain_dir, ft_subdir) — only the Full pretraining, best_s3 fine-tuned model
FT_RUNS = [
    ('outputs/pretraining',                'finetuning_exvivo_best_s3'),
]

print(f'Will run inference for {len(FT_RUNS)} fine-tuned models on ex-vivo data')


Will run inference for 1 fine-tuned models on ex-vivo data


In [3]:
# =========================================================================
# Load & preprocess ex-vivo data — one slice per odd src
# =========================================================================
def load_exvivo_stack(data_dir, slice_x=32):
    """Load ex-vivo .bin files for all odd src, build (S, H, W) stack.

    Only odd src indices that have BOTH a mask AND data file are
    loaded.  Missing src (e.g. src317) are filled with zeros.
    Returns (512, 384, 384) float32 array.
    """
    stack = np.zeros((512, 384, 384), dtype=np.float32)
    loaded, missing = 0, 0
    for src in range(1, 512, 2):  # odd only
        # Skip src that don't have a mask at all (e.g. src317)
        mask_path = os.path.join(data_dir, f'src{src}_mask.npy')
        if not os.path.exists(mask_path):
            missing += 1
            continue

        fname = f'slice{slice_x}_src{src}.bin'
        fpath = os.path.join(data_dir, fname)
        if not os.path.exists(fpath):
            print(f'  WARNING: {fname} not found — filling with zeros')
            missing += 1
            continue

        sub = np.fromfile(fpath, dtype=np.int16).reshape((384, 384)).astype(np.float32)
        vmax = np.abs(sub).max()
        if vmax > 0:
            sub /= vmax
        sub = np.clip(sub, -1.0, 1.0)
        stack[src - 1] = sub  # src1 → index 0
        loaded += 1
    print(f'  Loaded {loaded} slices, {missing} missing/zero-filled')
    return stack

SLICE_X = 32  # which cross-section to use per src
print(f'Loading ex-vivo data (slice {SLICE_X} per src) ...')
stack = load_exvivo_stack(DATA_DIR, slice_x=SLICE_X)
S, H, W = stack.shape
print(f'Preprocessed: {stack.shape}  range=[{stack.min():.3f}, {stack.max():.3f}]')
print(f'Nonzero slices: {(np.abs(stack).sum(axis=(1,2)) > 0).sum()}')

# DataLoader (reused for all models)
tensor_stack = torch.from_numpy(stack).unsqueeze(1)
dataset = TensorDataset(tensor_stack)
loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=4)

Loading ex-vivo data (slice 32 per src) ...
  Loaded 255 slices, 1 missing/zero-filled
Preprocessed: (512, 384, 384)  range=[-0.999, 1.000]
Nonzero slices: 255


In [4]:
# =========================================================================
# Run inference for one model
# =========================================================================
def run_inference_for_model(save_dir, model, loader, S, H):
    """Run inference and save results to *save_dir*."""
    os.makedirs(save_dir, exist_ok=True)

    all_masks = []
    t0 = time.time()
    with torch.no_grad():
        for (batch,) in loader:
            batch = batch.to(device)
            logits = model(batch)
            preds = logits.argmax(dim=1).cpu().numpy().astype(np.uint8)
            all_masks.append(preds)
    pred_masks = np.concatenate(all_masks, axis=0)
    elapsed = time.time() - t0

    # Boundary extraction
    boundary_matrix = np.full((H, S), np.nan, dtype=np.float32)
    for s in range(S):
        for row in range(H):
            ones = np.where(pred_masks[s, row, :] == 1)[0]
            if len(ones) > 0:
                boundary_matrix[row, s] = float(ones[0])

    # Save
    mask_3d = pred_masks.transpose(1, 2, 0)
    np.save(os.path.join(save_dir, 'pred_mask_3d.npy'), mask_3d)
    np.asfortranarray(mask_3d).tofile(os.path.join(save_dir, 'pred_mask_3d.bin'))
    np.save(os.path.join(save_dir, 'boundary_matrix.npy'), boundary_matrix)
    with open(os.path.join(save_dir, 'inference_time.txt'), 'w') as f:
        f.write(f'{elapsed:.2f}s  ({S} slices, {S/elapsed:.0f} slices/s)\n')

    return elapsed

In [5]:
# =========================================================================
# Run all
# =========================================================================
results = []
t_start = time.time()
INFERENCE_ROOT = os.path.join(_project_root, 'outputs', 'inference_results', 'exvivo')

for pretrain_rel, ft_subdir in FT_RUNS:
    pretrain_dir = os.path.join(_project_root, pretrain_rel)
    ft_dir = os.path.join(pretrain_dir, ft_subdir)
    ckpt_path = os.path.join(ft_dir, 'finetune_best.pth')

    # Subfolder name:  e.g. "pretraining_exvivo_latest", "pretraining_no_tof_exvivo_best_s3"
    exp_name = os.path.basename(pretrain_dir)
    strategy_short = ft_subdir.replace('finetuning_exvivo_', '')
    label = f'{exp_name}_{strategy_short}'

    save_dir = os.path.join(INFERENCE_ROOT, label)
    print(f'\n{"="*60}')
    print(f'  {label}  (ex-vivo)')
    print(f'  Checkpoint: {ckpt_path}')
    print(f'  Save to:    {save_dir}')
    print(f'{"="*60}')

    if not os.path.exists(ckpt_path):
        print(f'  !! SKIP: {ckpt_path} not found')
        results.append((label, 'SKIP'))
        continue

    # Read model config
    config_path = os.path.join(pretrain_dir, 'config.txt')
    if os.path.exists(config_path):
        with open(config_path) as f:
            cfg = json.load(f)
        base_ch = cfg.get('base_channel', 64)
        dropout = cfg.get('dropout_prob', 0.2)
        use_bn  = cfg.get('use_bn', True)
    else:
        base_ch, dropout, use_bn = 64, 0.2, True

    # Build & load model
    model = UNet(in_ch=1, base_ch=base_ch, num_classes=2,
                 dropout_prob=dropout, use_bn=use_bn).to(device)
    state = torch.load(ckpt_path, map_location=device, weights_only=True)
    model.load_state_dict(state)
    model.eval()

    # Run
    elapsed = run_inference_for_model(save_dir, model, loader, S, H)
    print(f'  DONE — {S} slices in {elapsed:.1f}s  ({S/elapsed:.0f} slices/s)')
    results.append((label, f'{elapsed:.1f}s'))

# =========================================================================
# Summary
# =========================================================================
print(f'\n{"="*60}')
print(f'  SUMMARY  (total: {(time.time()-t_start)/60:.0f} min)')
print(f'{"="*60}')
for label, status in results:
    print(f'  {label:45s}  {status}')


  pretraining_best_s3  (ex-vivo)
  Checkpoint: /data/projects/AgentWork/FAseg for github/outputs/pretraining/finetuning_exvivo_best_s3/finetune_best.pth
  Save to:    /data/projects/AgentWork/FAseg for github/outputs/inference_results/exvivo/pretraining_best_s3


/home/yifei-sun/anaconda3/envs/faseg_ablation/lib/python3.10/site-packages/torch/cuda/__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5090 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_89 sm_90 compute_90.
If you want to use the NVIDIA GeForce RTX 5090 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


  DONE — 512 slices in 2.7s  (192 slices/s)

  SUMMARY  (total: 0 min)
  pretraining_best_s3                            2.7s
